# Round 0: Python & SQL Coding — Ejercicios Prácticos

**Tiempo:** 60 min
**Objetivo:** SQL + Pandas + NumPy coding

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
import warnings
warnings.filterwarnings('ignore')
print('✅ Setup completo')

## Ejercicio 1: Pandas — Top N por grupo

In [ ]:
# Dataset de ventas
df = pd.DataFrame({
    'region': ['North','North','North','South','South','South','East','East','East'],
    'product': ['A','B','C','A','B','C','A','B','C'],
    'revenue': [100, 200, 150, 300, 250, 180, 120, 90, 210]
})

# TAREA: Top 2 productos por región
result = (
    df.groupby(['region', 'product'])['revenue']
    .sum()
    .reset_index()
    .sort_values(['region', 'revenue'], ascending=[True, False])
    .groupby('region')
    .head(2)
)
print(result)

## Ejercicio 2: NumPy — Estadísticas + Outliers

In [ ]:
prices = np.array([10, 12, 11, 13, 14, 9, 100, 11, 12, 10, 11, 13])

# Estadísticas
print(f'Mean: {np.mean(prices):.2f}')
print(f'Median: {np.median(prices):.2f}')
print(f'Std: {np.std(prices):.2f}')
print(f'P90: {np.percentile(prices, 90):.2f}')

# Z-score outliers
z_scores = np.abs(stats.zscore(prices))
print(f'\nOutliers (Z>2): {prices[z_scores > 2]}')

# IQR outliers
Q1, Q3 = np.percentile(prices, [25, 75])
IQR = Q3 - Q1
mask = (prices < Q1 - 1.5*IQR) | (prices > Q3 + 1.5*IQR)
print(f'Outliers (IQR): {prices[mask]}')

## Ejercicio 3: Data Cleaning Pipeline

In [ ]:
# Dataset sucio
df_dirty = pd.DataFrame({
    'customer_id': [1, 2, 2, 3, 4, 5],
    'age': [25, 30, 30, None, 45, 28],
    'income': [50000, 60000, 60000, 75000, None, 55000],
    'region': ['North', 'South', 'South', None, 'East', 'North'],
    'churn': [0, 1, 1, 0, 1, 0]
})

print('Original:')
print(df_dirty)
print(f'\nMissing:\n{df_dirty.isnull().sum()}')

# Limpiar
df_clean = df_dirty.copy()
df_clean = df_clean.drop_duplicates()
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['income'] = df_clean['income'].fillna(df_clean['income'].median())
df_clean['region'] = df_clean['region'].fillna(df_clean['region'].mode()[0])

print('\nLimpio:')
print(df_clean)
print(f'\nMissing: {df_clean.isnull().sum().sum()}')

## Ejercicio 4: Feature Engineering — RFM

In [ ]:
# Transacciones
transactions = pd.DataFrame({
    'customer_id': [1,1,1,2,2,3,3,3,3],
    'date': pd.to_datetime(['2024-01-01','2024-02-15','2024-03-20',
                            '2024-01-10','2024-03-25',
                            '2024-01-05','2024-01-20','2024-02-10','2024-03-30']),
    'amount': [100, 200, 150, 500, 300, 50, 75, 60, 80]
})

reference_date = transactions['date'].max()

features = transactions.groupby('customer_id').agg(
    recency=('date', lambda x: (reference_date - x.max()).days),
    frequency=('amount', 'count'),
    monetary=('amount', 'sum'),
    avg_amount=('amount', 'mean')
).reset_index()

print('RFM Features:')
print(features)

## Ejercicio 5: Sklearn Pipeline (sin data leakage)

In [ ]:
from sklearn.datasets import make_classification

# Dataset
X_raw, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_df = pd.DataFrame(X_raw, columns=[f'num_{i}' for i in range(8)] + ['cat_1_enc', 'cat_2_enc'])

# Split PRIMERO
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

# Pipeline
numerical_features = [f'num_{i}' for i in range(8)]

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numerical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit y eval
pipeline.fit(X_train, y_train)
test_acc = pipeline.score(X_test, y_test)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print(f'Test Accuracy: {test_acc:.4f}')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print('✅ Sin data leakage: split antes del preprocessing')

## SQL — Práctica (ejecutar en tu base de datos)

Estas queries puedes probarlas en: https://www.db-fiddle.com o https://sqliteonline.com

### Setup
```sql
CREATE TABLE orders (
    order_id INT,
    customer_id INT,
    amount DECIMAL(10,2),
    created_at DATE
);

INSERT INTO orders VALUES
(1, 101, 500, '2024-01-15'),
(2, 102, 200, '2024-02-20'),
(3, 101, 750, '2024-03-10'),
(4, 103, 300, '2024-01-25'),
(5, 102, 450, '2024-04-05'),
(6, 104, 1200, '2024-02-14'),
(7, 101, 100, '2023-12-01');
```

### Q0.1: Top clientes por revenue
```sql
SELECT customer_id, COUNT(*) num_orders,
       SUM(amount) total, AVG(amount) avg_amount
FROM orders
WHERE YEAR(created_at) = 2024
GROUP BY customer_id
ORDER BY total DESC
LIMIT 5;
```

### Q0.3: Window function
```sql
SELECT customer_id, amount,
       RANK() OVER (ORDER BY amount DESC) AS rank_by_amount,
       SUM(amount) OVER (ORDER BY created_at) AS running_total
FROM orders
WHERE YEAR(created_at) = 2024;
```

In [ ]:
# Ejercicio 1: Limpieza básica de lista
# Tienes esta lista con duplicados y None
datos = [1, 2, None, 3, 2, 4, None, 5, 1]

# Tarea:
# 1. Elimina los None
# 2. Elimina duplicados
# 3. Ordena de mayor a menor
# Resultado esperado: [5, 4, 3, 2, 1]

datos.discard(None)  # Elimina None
datos = set(datos)  # Elimina duplicados
datos = sorted(datos, reverse=True)  # Ordena de mayor a menor
print(datos)  # [5, 4, 3, 2, 1

In [ ]:
import pandas as pd
# Ejercicio 2: Diccionario → DataFrame
# Tienes este diccionario
ventas = {
    'enero':  [100, 200, 150],
    'febrero': [120, 180, 160],
    'marzo':  [140, 220, 170]
}
# Filas = vendedores: ['Ana', 'Bob', 'Carlos']

# Tarea:
# 1. Crear DataFrame con vendedores como índice
# 2. Agregar columna 'total' con suma de los 3 meses
# 3. Ordenar por total descendente

df_ventas = pd.DataFrame(ventas, index=['Ana', 'Bob', 'Carlos'])
df_ventas['total'] = df_ventas.sum(axis=1)
df_ventas = df_ventas.sort_values('total', ascending=False)

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'edad':     [25, np.nan, 30, 45, np.nan],
    'ingresos': [50000, 60000, np.nan, 80000, 55000],
    'ciudad':   ['Bogotá', 'Medellín', np.nan, 'Cali', 'Bogotá']
})

# Tarea:
# 1. Imputa edad con la mediana
# 2. Imputa ingresos con la media
# 3. Imputa ciudad con la moda
# 4. Verifica que no queden NaNs

df['edad'] = df['edad'].fillna(df['edad'].median())
df['ingresos'] = df['ingresos'].fillna(df['ingresos'].mean())
df['ciudad'] = df['ciudad'].fillna(df['ciudad'].mode()[0])      

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nombre': ['ana garcia', 'BOB SMITH', 'carlos LOPEZ'],
    'salario': [3000000, 5000000, 4500000],
    'años_exp': [2, 8, 5]
})

# Tarea:
# 1. Normaliza 'nombre' a Title Case (Ana Garcia)
# 2. Crea columna 'nivel':
#    años_exp < 3 → 'Junior'
#    años_exp < 7 → 'Mid'
#    años_exp >= 7 → 'Senior'
# 3. Crea columna 'salario_ajustado':
#    Junior → salario * 1.1
#    Mid → salario * 1.15
#    Senior → salario * 1.2

df['nombre'] = df['nombre'].str.title() 
df['nivel'] = pd.cut(df['años_exp'], bins=[-1, 3, 7, np.inf], labels=['Junior', 'Mid', 'Senior'])
df['salario_ajustado'] = df.apply(lambda row: row['salario'] * (1.1 if row['nivel'] == 'Junior' else 1.15 if row['nivel'] == 'Mid' else 1.2), axis=1)

In [1]:
numbers = [4, 1, 7, 3, 9, 2]

def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

sorted_numbers = bubble_sort(numbers)

In [3]:
sorted_numbers

def second_highest(arr):
    unique = list(set(arr))
    unique.sort(reverse=True)
    return unique[1]

numbers = [4, 1, 7, 3, 9, 2]
print(second_highest(numbers))  # 7


7


In [5]:
def isPalindrome(s):
    cleaned = ''.join(c.lower() for c in s if c.isalnum())
    return cleaned == cleaned[::-1]

isPalindrome("hello")
isPalindrome("racecar")

True